In [8]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

from sentence_transformers import SentenceTransformer

In [3]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(text):
    """Nettoie un texte : minuscules, supprime ponctuation, gère les NaN."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

class RecommendationEmploi:
    def __init__(self):
        # Téléchargement au premier appel (cache local ensuite)
        self.model_emb = SentenceTransformer("all-MiniLM-L6-v2")
        self.df = None

    def fit(self, df: pd.DataFrame):
        required_cols = ["poste", "competence_clean", "region_clean", "experience"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Colonnes manquantes dans le DataFrame : {missing}")

        df = df.copy()  # Évite les warnings pandas
        df["job_profile"] = (
            df["poste"].fillna("") + " " + 
            df["competence_clean"].fillna("") + " " + 
            df["region_clean"].fillna("")
        )

        # 🔧 CORRECTION : list() transforme la matrice (n, 384) en [vec1, vec2, ...]
        df["embedding"] = list(self.model_emb.encode(df["job_profile"].tolist(), show_progress_bar=False))
        
        self.df = df

    # -----------------------------
    # SCORING FUNCTIONS
    # -----------------------------
    def skill_score(self, user_emb, job_emb):
        return cosine_similarity([user_emb], [job_emb])[0][0]

    def experience_score(self, user_exp, job_exp):
        diff = abs(user_exp - job_exp)
        return max(0, 1 - diff / 10)

    def region_score(self, user_region, job_region):
        return 1.0 if user_region == job_region else 0.3

    def hybrid_score(self, skill, exp, region):
        return 0.6 * skill + 0.25 * exp + 0.15 * region

    # -----------------------------
    # SKILLS GAP
    # -----------------------------
    def missing_skills(self, user_skills, job_skills):
        user_set = set(user_skills.split())
        job_set = set(job_skills.split())
        return list(job_set - user_set)

    # -----------------------------
    # RECOMMENDATION
    # -----------------------------
    def recommend(self, skills, region, experience, top_k=5):
        user_text = clean_text(skills)
        user_region = clean_text(region)
        user_emb = self.model_emb.encode(user_text)

        results = []
        for _, row in self.df.iterrows():
            s_score = self.skill_score(user_emb, row["embedding"])
            e_score = self.experience_score(experience, row["experience"])
            r_score = self.region_score(user_region, row["region_clean"])

            final_score = self.hybrid_score(s_score, e_score, r_score)

            results.append({
                "poste": row["poste"],
                "region": row["region_clean"],
                "score": round(final_score, 4),
                "missing_skills": self.missing_skills(user_text, row["competence_clean"])
            })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_k]

In [5]:
# 1. Création d'un DataFrame d'exemple
data = {
    "poste": ["Data Scientist", "Développeur Web", "Analyste RH"],
    "competence_clean": ["python machine learning sql", "javascript react css", "recrutement droit social paie"],
    "region_clean": ["Île-de-France", "Nouvelle-Aquitaine", "Île-de-France"],
    "experience": [3, 2, 5]
}
df_offres = pd.DataFrame(data)

# 2. Initialisation & apprentissage
reco = RecommendationEmploi()
reco.fit(df_offres)

# 3. Recommandation pour un profil
resultats = reco.recommend(
    skills="python data analyse machine learning",
    region="ile de france",
    experience=2,
    top_k=2
)

# 4. Affichage
for r in resultats:
    print(f"📌 {r['poste']} | Score: {r['score']} | Région: {r['region']}")
    print(f"   ⚠️ Compétences à acquérir : {r['missing_skills']}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1573.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📌 Data Scientist | Score: 0.6367999911308289 | Région: Île-de-France
   ⚠️ Compétences à acquérir : ['sql']

📌 Développeur Web | Score: 0.2554999887943268 | Région: Nouvelle-Aquitaine
   ⚠️ Compétences à acquérir : ['css', 'react', 'javascript']



In [20]:
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------
# CLEANING
# -----------------------------
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def deduplicate_skills(text):
    if pd.isna(text):
        return ""
    words = text.split()
    return " ".join(dict.fromkeys(words))  # supprime doublons en gardant ordre


# -----------------------------
# MODEL
# -----------------------------
class RecommendationEmploi:

    def __init__(self):
        self.model_emb = SentenceTransformer("all-MiniLM-L6-v2")
        self.df = None
        self.embeddings = None  # ⚡ cache embeddings pour vitesse

    # -----------------------------
    # FIT
    # -----------------------------
    def fit(self, df: pd.DataFrame):

        required_cols = ["poste", "competence_clean", "region_clean", "experience"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Colonnes manquantes : {missing}")

        df = df.copy()

        # 🔥 nettoyage complet
        df["competence_clean"] = df["competence_clean"].apply(clean_text)
        df["competence_clean"] = df["competence_clean"].apply(deduplicate_skills)
        df["region_clean"] = df["region_clean"].apply(clean_text)

        # 🔥 création profil
        df["job_profile"] = (
            df["poste"].fillna("") + " " +
            df["competence_clean"] + " " +
            df["region_clean"]
        )

        # ⚡ encoding batch (rapide)
        self.embeddings = self.model_emb.encode(
            df["job_profile"].tolist(),
            show_progress_bar=False
        )

        self.df = df.reset_index(drop=True)

        return self

    # -----------------------------
    # SCORES
    # -----------------------------
    def experience_score(self, user_exp, job_exp):
        diff = abs(user_exp - job_exp)
        return max(0, 1 - diff / 10)

    def region_score(self, user_region, job_regions):
        return np.where(job_regions == user_region, 1.0, 0.3)

    # -----------------------------
    # SKILL GAP
    # -----------------------------
    def missing_skills(self, user_skills, job_skills):
        user_set = set(user_skills.split())
        job_set = set(job_skills.split())
        return list(job_set - user_set)

    # -----------------------------
    # RECOMMENDATION (OPTIMISÉE)
    # -----------------------------
    def recommend(self, skills, region, experience, top_k=5):

        user_text = clean_text(skills)
        user_region = clean_text(region)

        # embedding user
        user_emb = self.model_emb.encode(user_text)

        # ⚡ similarity vectorisée
        skill_scores = cosine_similarity(
            [user_emb],
            self.embeddings
        )[0]

        # ⚡ scores expérience
        exp_scores = np.array([
            self.experience_score(experience, e)
            for e in self.df["experience"]
        ])

        # ⚡ score région vectorisé
        region_scores = self.region_score(
            user_region,
            self.df["region_clean"].values
        )

        # 🔥 score final
        final_scores = (
            0.6 * skill_scores +
            0.25 * exp_scores +
            0.15 * region_scores
        )

        # ⚡ top K rapide
        top_idx = np.argsort(final_scores)[::-1][:top_k]

        results = []
        for idx in top_idx:
            row = self.df.iloc[idx]

            results.append({
                "poste": row["poste"],
                "region": row["region_clean"],
                "score": round(float(final_scores[idx]), 4),
                "missing_skills": self.missing_skills(
                    user_text,
                    row["competence_clean"]
                )
            })

        return results

In [9]:
import pandas as pd
import psycopg2

class Visualisation:
    def __init__(self, host, port, database, user, password):
        self.conn_params = {
            "host": host,
            "port": port,
            "database": database,
            "user": user,
            "password": password
        }
        self.conn = None
        self._connect()

    def _connect(self):
        try:
            self.conn = psycopg2.connect(**self.conn_params)
            print("Connexion réussie à la base de données PostgreSQL")
        except Exception as e:
            print(f" Erreur de connexion : {e}")
            self.conn = None

    def get_data(self, table_name):
        if not self.conn:
            print("Aucune connexion active.")
            return None
        
        # Sécurité basique contre l'injection SQL sur le nom de table
        if not table_name.replace("_", "").isalnum():
            print(" Nom de table invalide.")
            return None
            
        try:
            query = f'SELECT * FROM "{table_name}"'
            df = pd.read_sql_query(query, self.conn)
            print(f" {len(df)} lignes récupérées depuis '{table_name}'")
            return df
        except Exception as e:
            print(f" Erreur lors de la requête : {e}")
            return None

    def close(self):
        if self.conn:
            self.conn.close()
            print("Connexion fermée.")

In [10]:
def parse_experience(exp):
    """
    Convertit une chaîne d'expérience en valeur numérique (années)
    """
    if not isinstance(exp, str):
        return 0.0
    
    exp = exp.lower().strip()
    
    mapping = {
        "etudiant": 0.0,
        "jeune diplômé": 0.3,
        "jeune diplômé et plus": 0.6,
        "débutant < 2 ans": 1.0,
        "débutant < 2 ans et plus": 1.6,
        "expérience entre 2 ans et 5 ans": 2.0,
        "expérience entre 2 ans et 5 ans et plus": 3.0,
        "expérience entre 5 ans et 10 ans": 5.0,
        "expérience > 10 ans": 10.0
    }
    
    # Nettoyer la chaîne pour correspondre au mapping
    # Remplacer les caractères spéciaux
    exp = exp.replace("'", "")
    
    # Chercher la correspondance exacte
    if exp in mapping:
        return mapping[exp]
    
    # Gestion des cas particuliers avec des mots-clés
    if "jeune diplômé" in exp:
        if "plus" in exp:
            return 0.6
        return 0.3
    elif "débutant" in exp or "etudiant" in exp and "2 ans" in exp:
        if "plus" in exp:
            return 1.6
        return 1.0
    elif "2 ans et 5 ans" in exp:
        if "plus" in exp:
            return 3.0
        return 2.0
    elif "5 ans et 10 ans" in exp:
        return 5.0
    elif "10 ans" in exp:
        return 10.0
    elif "etudiant" in exp:
        return 0.0
    
    # Valeur par défaut
    return 0.0

In [11]:
# 1. Initialisation avec les paramètres de connexion
viz = Visualisation(
    host="localhost",
    port=5493,
    database="datawarehouse",
    user="admin",
    password="admin_pwd"
)

# 2. Récupération des données
df = viz.get_data("offres_emploi_ml")

Connexion réussie à la base de données PostgreSQL
 35792 lignes récupérées depuis 'offres_emploi_ml'


/tmp/ipykernel_197178/4236701334.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [14]:
df["experience_clean"] = df["experience"].apply(parse_experience)
df.columns

Index(['entreprise', 'poste', 'competence', 'formation_clean', 'niveau_etude',
       'contrat', 'experience', 'region', 'date_de_publication',
       'experience_clean'],
      dtype='str')

In [ ]:
df = viz.get_data("offres_emploi_ml")

def precessing(dt):
    dt["experience_clean"] = dt["experience"].apply(parse_experience)
    grouped = dt.groupby(["poste", "region", "experience_clean"])["competence"] \
                    .apply(lambda x: " ".join(x)).dropna().reset_index()
    data_grouped = grouped.rename(columns={
    "competence": "competence_clean",
    "region": "region_clean",
    "experience_clean": "experience"
    })
    return data_grouped



 35792 lignes récupérées depuis 'offres_emploi_ml'


/tmp/ipykernel_197178/4236701334.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [34]:
data1 = precessing(df)
data1

,poste,region_clean,experience,competence_clean
0,Assistant Comptable (Stagiaire) - Dakar,Dakar,1.6,Comptabilité Finance Gestion Rapprochement Ban...
1,Chargé de Clientèle Bilingue (Français-Anglai...,Dakar,0.0,Vente Vente Vente Vente Vente Vente Vente Vent...
2,Chargé de Clientèle Bilingue (Français-Anglai...,Dakar,0.6,Vente Vente Vente Vente Vente Vente Vente Vent...
3,ANGULAR Developer (M/F),Dakar,5.0,BOOTSTRAP BOOTSTRAP BOOTSTRAP CSS3 CSS3 CSS3 C...
4,ANGULAR Developer (M/F),Diourbel,5.0,BOOTSTRAP BOOTSTRAP BOOTSTRAP CSS3 CSS3 CSS3 C...
...,...,...,...,...
490,Z/OS System Engineer Senior Storage(H/F),Fatick,5.0,CICS CICS CICS DB2 DB2 DB2 IBM IBM IBM JCL JCL...
491,Z/OS System Engineer Senior Storage(H/F),Kaffrine,5.0,CICS CICS CICS DB2 DB2 DB2 IBM IBM IBM JCL JCL...
492,Z/OS System Engineer Senior Storage(H/F),Kaolack,5.0,CICS CICS CICS DB2 DB2 DB2 IBM IBM IBM JCL JCL...
493,Z/OS System Engineer Senior Storage(H/F),Kolda,5.0,CICS CICS CICS DB2 DB2 DB2 IBM IBM IBM JCL JCL...


In [33]:


# -----------------------------
# 1. Charger les données
# -----------------------------


# ⚠️ adapte si besoin


# -----------------------------
# 2. Initialiser + entraîner
# -----------------------------
model = RecommendationEmploi()
model.fit(data1)

print("✅ Modèle entraîné")

# -----------------------------
# 3. Test prédiction
# -----------------------------
skills = "python machine learning sql java"
region = "dakar"
experience = 2

results = model.recommend(
    skills=skills,
    region=region,
    experience=experience,
    top_k=5
)

# -----------------------------
# 4. Affichage
# -----------------------------
print("\n🎯 Recommandations :\n")

for i, r in enumerate(results, 1):
    print(f"{i}. Poste : {r['poste']}")
    print(f"   Région : {r['region']}")
    print(f"   Score  : {r['score']}")
    print(f"   Skills manquants : {r['missing_skills']}")
    print("-" * 40)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1774.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle entraîné

🎯 Recommandations :

1. Poste : Stagiaire Développeur Web - Dakar
   Région : dakar
   Score  : 0.5838
   Skills manquants : ['css', 'mongodb', 'mysql', 'javascript', 'postgresql', 'html']
----------------------------------------
2. Poste : Test Automation Engineer (M/F)
   Région : dakar
   Score  : 0.556
   Skills manquants : ['jenkins', 'gitlab', 'shell', 'api', 'matrix', 'cvs', 'linux', 'javascript', 'mysql', 'oracle', 'git']
----------------------------------------
3. Poste : GENESYS Architect (M/F)
   Région : dakar
   Score  : 0.5288
   Skills manquants : ['paas', 'bash', 'powershell', 'cvs', 'genesys', 'vmware', 'javascript']
----------------------------------------
4. Poste : PYTHON Developer (M/F)
   Région : dakar
   Score  : 0.526
   Skills manquants : ['api', 'django', 'css', 'agile', 'cvs', 'tornado', 'html', 'scrum']
----------------------------------------
5. Poste : QA Tester (M/F)
   Région : dakar
   Score  : 0.5215
   Skills manquants : ['app', 'a